In [1]:
#Q1
import pandas as pd

fixed_entries = [
    {"question": "what is the annual fee", "answer": "The annual fee is Rs 500.",
     "keywords": "fee cost price charge", "category": "billing"},
    {"question": "how to reset password", "answer": "Go to Settings > Reset Password.",
     "keywords": "password reset login", "category": "account"},
    {"question": "what are your working hours", "answer": "We are open 9 AM to 5 PM.",
     "keywords": "hours timing open time", "category": "general"},
    {"question": "how can i pay the fee", "answer": "You can pay via UPI, card, or net banking.",
     "keywords": "pay payment upi fee", "category": "billing"},
]

# Roll number: 1024160136 -> last two digits: 3, 6
# digit 3 -> category[3 % 3] = category[0] = "billing"
# digit 6 -> category[6 % 3] = category[0] = "billing"

own_entries = [
    {"question": "how do i get a refund", "answer": "Refunds are processed within 5-7 business days.",
     "keywords": "refund money return", "category": "billing"},
    {"question": "can i get an invoice for my payment", "answer": "You can download your invoice from the Billing section.",
     "keywords": "invoice receipt bill", "category": "billing"},
]

faq_entries = fixed_entries + own_entries
df = pd.DataFrame(faq_entries)
print(df)

                              question  \
0               what is the annual fee   
1                how to reset password   
2          what are your working hours   
3                how can i pay the fee   
4                how do i get a refund   
5  can i get an invoice for my payment   

                                              answer                keywords  \
0                          The annual fee is Rs 500.   fee cost price charge   
1                   Go to Settings > Reset Password.    password reset login   
2                          We are open 9 AM to 5 PM.  hours timing open time   
3         You can pay via UPI, card, or net banking.     pay payment upi fee   
4    Refunds are processed within 5-7 business days.     refund money return   
5  You can download your invoice from the Billing...    invoice receipt bill   

  category  
0  billing  
1  account  
2  general  
3  billing  
4  billing  
5  billing  


In [2]:
#Q2
def score_query(query, df):
    query_words = set(query.lower().split())
    results = []

    for idx, row in df.iterrows():
        keyword_words = set(row["keywords"].lower().split())
        score = len(query_words & keyword_words)
        if score > 0:
            results.append({
                "question": row["question"],
                "answer": row["answer"],
                "category": row["category"],
                "score": score
            })

    results_df = pd.DataFrame(results)
    if not results_df.empty:
        results_df = results_df.sort_values(by="score", ascending=False).reset_index(drop=True)
    return results_df

# Example usage
result = score_query("how do i pay the fee", df)
print(result)

                 question                                      answer  \
0   how can i pay the fee  You can pay via UPI, card, or net banking.   
1  what is the annual fee                   The annual fee is Rs 500.   

  category  score  
0  billing      2  
1  billing      1  


In [3]:
#Q3
def same_category(category_name, df):
    return df[df["category"] == category_name].reset_index(drop=True)

# Call it using the category of one of the personalized entries (e.g. "billing")
category_result = same_category("billing", df)
print(category_result)

                              question  \
0               what is the annual fee   
1                how can i pay the fee   
2                how do i get a refund   
3  can i get an invoice for my payment   

                                              answer               keywords  \
0                          The annual fee is Rs 500.  fee cost price charge   
1         You can pay via UPI, card, or net banking.    pay payment upi fee   
2    Refunds are processed within 5-7 business days.    refund money return   
3  You can download your invoice from the Billing...   invoice receipt bill   

  category  
0  billing  
1  billing  
2  billing  
3  billing  


In [4]:
#Q4
# Pick one entry to update (e.g. index 0)
entry_index = 0
new_keyword = input("Enter a new keyword to add: ")

df.at[entry_index, "keywords"] = df.at[entry_index, "keywords"] + " " + new_keyword
print("Updated entry:\n", df.iloc[entry_index])

# Save the updated DataFrame to CSV
roll_number = "1024160136"
filename = f"{roll_number}_faq_data.csv"
df.to_csv(filename, index=False)
print(f"\nSaved to {filename}")

Enter a new keyword to add: discount
Updated entry:
 question            what is the annual fee
answer           The annual fee is Rs 500.
keywords    fee cost price charge discount
category                           billing
Name: 0, dtype: object

Saved to 1024160136_faq_data.csv


In [5]:
#Q5
category_counts = df.groupby("category").size()
print("FAQ entries per category:\n", category_counts)

FAQ entries per category:
 category
account    1
billing    4
general    1
dtype: int64


In [6]:
#Q6
def score_query_with_ties(query, df):
    query_words = set(query.lower().split())
    results = []

    for idx, row in df.iterrows():
        keyword_words = set(row["keywords"].lower().split())
        score = len(query_words & keyword_words)
        if score > 0:
            results.append({
                "question": row["question"],
                "answer": row["answer"],
                "category": row["category"],
                "score": score
            })

    if not results:
        print("No matches found.")
        return pd.DataFrame()

    results_df = pd.DataFrame(results)
    max_score = results_df["score"].max()
    top_matches = results_df[results_df["score"] == max_score].reset_index(drop=True)

    if len(top_matches) > 1:
        print(f"Tie detected! {len(top_matches)} entries share the highest score of {max_score}:")
    else:
        print(f"Best match found with score {max_score}:")

    print(top_matches)
    return top_matches

# Example query that produces a tie (matches both "fee" entries)
print("--- Query that ties ---")
score_query_with_ties("fee", df)

# Example query that does NOT tie
print("\n--- Query that does not tie ---")
score_query_with_ties("password login", df)

--- Query that ties ---
Tie detected! 2 entries share the highest score of 1:
                 question                                      answer  \
0  what is the annual fee                   The annual fee is Rs 500.   
1   how can i pay the fee  You can pay via UPI, card, or net banking.   

  category  score  
0  billing      1  
1  billing      1  

--- Query that does not tie ---
Best match found with score 2:
                question                            answer category  score
0  how to reset password  Go to Settings > Reset Password.  account      2


,question,answer,category,score
0,how to reset password,Go to Settings > Reset Password.,account,2
